In [1]:
import pandas as pd
import json

# ── Load & prep ────────────────────────────────────────────────────────────────
df = pd.read_csv("master.csv")
parts = df["PR Link"].str.split("/", expand=True)
df["app"] = parts[3] + "_" + parts[4]

# ── Load app risk flags ────────────────────────────────────────────────────────
with open("app_flags.json", "r") as f:
    app_flags = json.load(f)

RISK_ORDER = {
    "safe": 0,
    "yellow": 1,
    "red": 2,
    "dark_red": 3,
}

RISK_LEVELS = ["safe", "yellow", "red", "dark_red"]
DEFAULT_RISK = "dark_red"

def get_app_risk(app_name):
    app_info = app_flags.get("apps", {}).get(app_name, {})
    risk = app_info.get("risk", DEFAULT_RISK)

    if risk not in RISK_ORDER:
        risk = DEFAULT_RISK

    return risk

df["risk"] = df["app"].apply(get_app_risk)
df["risk_rank"] = df["risk"].apply(lambda r: RISK_ORDER[r])

# ── App exception list ─────────────────────────────────────────────────────────
EXCLUDED_APPS = {
    "code-dot-org_code-dot-org",
    # "another-owner_another-repo",
    # "problematic-app-name_here",
}

df = df[~df["app"].isin(EXCLUDED_APPS)]

df = df[df["Include?"] == 1]

# Shuffle for tie-breaking randomness
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop rows missing either required annotation
df = df[df["Broader issue type"].notna() & df["User Demographic"].notna()].reset_index(drop=True)

def parse_field(value):
    """Split a comma-separated field into a cleaned list."""
    return [x.strip() for x in str(value).split(",") if x.strip()]

TARGET = 50
MIN_APPS = 10

# Full target coverage from all eligible, non-excluded PRs
all_issue_types = set(it for v in df["Broader issue type"] for it in parse_field(v))
all_user_demos  = set(d  for v in df["User Demographic"] for d in parse_field(v))


def run_sampling(candidate_df):
    """
    Run the same coverage-based sampler on a candidate subset of apps.
    candidate_df should already be filtered by risk level.
    """

    sampled_rows = []
    sample_issue_types = set()
    sample_user_demos = set()
    sample_apps = set()
    used_indices = set()

    # Prefer lower-risk apps first, then tie-break using shuffled order
    candidate_df = candidate_df.sort_values(
        by=["risk_rank"],
        ascending=True
    )

    # ── Phase 1: coverage-driven selection ────────────────────────────────────
    for prefer_known_app in [True, False]:
        if len(sampled_rows) >= TARGET:
            break

        for i, row in candidate_df.iterrows():
            if i in used_indices:
                continue
            if len(sampled_rows) >= TARGET:
                break

            issue_types = parse_field(row["Broader issue type"])
            user_demos  = parse_field(row["User Demographic"])
            app         = row["app"]

            adds_new_issue = any(it not in sample_issue_types for it in issue_types)
            adds_new_demo  = any(d  not in sample_user_demos  for d  in user_demos)

            if not (adds_new_issue or adds_new_demo):
                continue

            # App-minimizing rule
            if prefer_known_app and app not in sample_apps and len(sample_apps) > 0:
                continue

            # Reserve enough remaining slots to reach MIN_APPS
            remaining_slots = TARGET - len(sampled_rows)
            missing_apps = max(0, MIN_APPS - len(sample_apps))

            if app in sample_apps and remaining_slots <= missing_apps:
                continue

            sampled_rows.append(row)
            used_indices.add(i)
            sample_issue_types.update(issue_types)
            sample_user_demos.update(user_demos)
            sample_apps.add(app)

    # ── Phase 1.5: force minimum app coverage ─────────────────────────────────
    if len(sample_apps) < MIN_APPS:
        for i, row in candidate_df.iterrows():
            if i in used_indices:
                continue
            if len(sampled_rows) >= TARGET:
                break
            if len(sample_apps) >= MIN_APPS:
                break

            app = row["app"]

            if app in sample_apps:
                continue

            sampled_rows.append(row)
            used_indices.add(i)
            sample_issue_types.update(parse_field(row["Broader issue type"]))
            sample_user_demos.update(parse_field(row["User Demographic"]))
            sample_apps.add(app)

    # ── Phase 2: fill-up to TARGET ─────────────────────────────────────────────
    if len(sampled_rows) < TARGET:
        for prefer_known_app in [True, False]:
            if len(sampled_rows) >= TARGET:
                break

            for i, row in candidate_df.iterrows():
                if i in used_indices:
                    continue
                if len(sampled_rows) >= TARGET:
                    break

                app = row["app"]

                if prefer_known_app and app not in sample_apps:
                    continue

                sampled_rows.append(row)
                used_indices.add(i)
                sample_issue_types.update(parse_field(row["Broader issue type"]))
                sample_user_demos.update(parse_field(row["User Demographic"]))
                sample_apps.add(app)

    sampled_df = pd.DataFrame(sampled_rows, columns=candidate_df.columns).reset_index(drop=True)

    return sampled_df, sample_issue_types, sample_user_demos, sample_apps


# ── Risk-aware sampling ────────────────────────────────────────────────────────
best_result = None
chosen_max_risk = None

for max_risk in RISK_LEVELS:
    max_rank = RISK_ORDER[max_risk]

    candidate_df = df[df["risk_rank"] <= max_rank].copy()

    sampled_df, sample_issue_types, sample_user_demos, sample_apps = run_sampling(candidate_df)

    has_target_size = len(sampled_df) == TARGET
    has_min_apps = len(sample_apps) >= MIN_APPS
    has_all_issues = sample_issue_types == all_issue_types
    has_all_demos = sample_user_demos == all_user_demos

    print(f"\nTrying apps up to risk level: {max_risk}")
    print(f"  Candidate PRs       : {len(candidate_df)}")
    print(f"  Sample size         : {len(sampled_df)}")
    print(f"  Distinct apps       : {len(sample_apps)}")
    print(f"  Issue coverage      : {len(sample_issue_types)}/{len(all_issue_types)}")
    print(f"  User demo coverage  : {len(sample_user_demos)}/{len(all_user_demos)}")

    best_result = (sampled_df, sample_issue_types, sample_user_demos, sample_apps)
    chosen_max_risk = max_risk

    if has_target_size and has_min_apps and has_all_issues and has_all_demos:
        break


sampled_df, sample_issue_types, sample_user_demos, sample_apps = best_result

# ── Clean helper columns before saving ─────────────────────────────────────────
sampled_df = sampled_df.drop(columns=["risk_rank"], errors="ignore").reset_index(drop=True)

# ── Coverage report ────────────────────────────────────────────────────────────
print("\n── Final sampling result ───────────────────────────────────────────────")
print(f"Chosen max risk     : {chosen_max_risk}")
print(f"Excluded apps       : {sorted(EXCLUDED_APPS)}")
print(f"Sample size         : {len(sampled_df)}")
print(f"Distinct apps       : {len(sample_apps)}")
print(f"Issue type coverage : {len(sample_issue_types)}/{len(all_issue_types)}"
      f" ({100*len(sample_issue_types)/len(all_issue_types):.1f}%)")
print(f"User demo coverage  : {len(sample_user_demos)}/{len(all_user_demos)}"
      f" ({100*len(sample_user_demos)/len(all_user_demos):.1f}%)")

print(f"\nApps in sample:\n{sorted(sample_apps)}")
print(f"\nIssue types covered:\n{sorted(sample_issue_types)}")
print(f"\nUser demographics covered:\n{sorted(sample_user_demos)}")

print("\nRisk breakdown:")
print(sampled_df["risk"].value_counts())

# ── Per-app breakdown ──────────────────────────────────────────────────────────
print("\n── Per-app breakdown ──────────────────────────────────────────────────")
app_issues = {}
app_demos  = {}
app_risks  = {}

for _, row in sampled_df.iterrows():
    app = row["app"]
    app_issues.setdefault(app, set()).update(parse_field(row["Broader issue type"]))
    app_demos.setdefault(app,  set()).update(parse_field(row["User Demographic"]))
    app_risks[app] = row["risk"]

for app in sorted(app_issues, key=lambda a: len(app_issues[a]), reverse=True):
    issues = sorted(app_issues[app])
    demos  = sorted(app_demos[app])
    risk   = app_risks[app]

    print(f"\n{app} [{risk}]")
    print(f"  Issue types ({len(issues)}): {', '.join(issues)}")
    print(f"  User demographics ({len(demos)}): {', '.join(demos)}")

sampled_df.to_csv("sampled_prs.csv", index=False)
print("\nSaved to sampled_prs.csv")


Trying apps up to risk level: safe
  Candidate PRs       : 78
  Sample size         : 50
  Distinct apps       : 13
  Issue coverage      : 8/8
  User demo coverage  : 4/4

── Final sampling result ───────────────────────────────────────────────
Chosen max risk     : safe
Excluded apps       : ['code-dot-org_code-dot-org']
Sample size         : 50
Distinct apps       : 13
Issue type coverage : 8/8 (100.0%)
User demo coverage  : 4/4 (100.0%)

Apps in sample:
['DDDEastMidlandsLimited_dddem-web', 'Leaflet_Leaflet', 'ONSdigital_design-system', 'Refinitiv_refinitiv-ui', 'cds-snc_digital-canada-ca', 'cylc_cylc-ui', 'dhis2_ui', 'gympass_yoga', 'inclusive-design_wecount.inclusivedesign.ca', 'maplibre_maputnik', 'mdo_github-buttons', 'twbs_bootstrap', 'unl_wdntemplates']

Issue types covered:
['Color contrast', 'Focus management', 'Heading and content structure', 'Keyboard navigation', 'Motion sensitivity', 'Screen reader information', 'Target size', 'Text presentation']

User demographics cov

In [3]:
import pandas as pd
df4=pd.read_csv("/media/safwat/Academics/GMU/Accessibility_ProgramRepair/A11y_Repair_Benchmark_4/sampled_prs.csv")
df4_prID=set(df4['PR Id'].tolist())
df5=pd.read_csv("/media/safwat/Academics/GMU/Accessibility_ProgramRepair/A11y_Repair_Benchmark_5/sampled_prs.csv")
df5_prID=set(df5['PR Id'].tolist())





In [6]:
for pr_id in df4_prID:
    if pr_id not in df5_prID:
        print(f"PR {pr_id} is missing in Benchmark 5")  

In [7]:
for pr_id in df5_prID:
    if pr_id not in df4_prID:
        print(f"PR {pr_id} is missing in Benchmark 4")  

PR 946583554 is missing in Benchmark 4
PR 613653257 is missing in Benchmark 4
PR 462831247 is missing in Benchmark 4
PR 1845281685 is missing in Benchmark 4
PR 1336005533 is missing in Benchmark 4
PR 1154179361 is missing in Benchmark 4
PR 1821851175 is missing in Benchmark 4
PR 2207657543 is missing in Benchmark 4
PR 950782151 is missing in Benchmark 4
PR 1163893375 is missing in Benchmark 4
PR 461626704 is missing in Benchmark 4
PR 798401620 is missing in Benchmark 4
PR 3068230238 is missing in Benchmark 4
PR 321971039 is missing in Benchmark 4
PR 65070942 is missing in Benchmark 4
PR 3068452196 is missing in Benchmark 4
PR 3019067117 is missing in Benchmark 4
PR 2173547118 is missing in Benchmark 4
PR 915924080 is missing in Benchmark 4
PR 2214849521 is missing in Benchmark 4
PR 1061148272 is missing in Benchmark 4
PR 1029860470 is missing in Benchmark 4
PR 500942454 is missing in Benchmark 4
PR 802213497 is missing in Benchmark 4
PR 503646332 is missing in Benchmark 4


In [9]:
df5_apps=set(df5['app'].tolist())
df5_apps


{'DDDEastMidlandsLimited_dddem-web',
 'Leaflet_Leaflet',
 'ONSdigital_design-system',
 'Refinitiv_refinitiv-ui',
 'cds-snc_digital-canada-ca',
 'cylc_cylc-ui',
 'dhis2_ui',
 'gympass_yoga',
 'inclusive-design_wecount.inclusivedesign.ca',
 'maplibre_maputnik',
 'mdo_github-buttons',
 'twbs_bootstrap',
 'unl_wdntemplates'}